# Notebook 2: AI Function Creation

## Objective

Build a CLASSIFY_SUPPORT_TICKET() AI function using **SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION()** — the managed stored procedure.

| Approach | Tradeoff |
|----------|----------|
| Raw CREATE FUNCTION DDL | Works, but no tracking, no EVALUATE/OPTIMIZE integration |
| CALL SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION() | Tagged, tracked, integrated with eval and optimize workflows |

---

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
try:
    state = session.sql("SELECT NOTEBOOK_ID FROM AI_STUDIO_DEMO.PUBLIC.DEMO_STATE WHERE NOTEBOOK_ID = 'setup'").collect()
    if not state: raise Exception()
    print('Prerequisite check passed: Notebook 1 (Setup) completed.')
except:
    print('WARNING: Please run Notebook 1 (Setup & Sample Data) first!')

In [ ]:
USE DATABASE AI_STUDIO_DEMO;
USE SCHEMA PUBLIC;

## V1: First Attempt

9 positional parameters: FUNCTION_NAME, MODEL, SYSTEM_PROMPT ($$), USER_PROMPT_TEMPLATE ($$), INPUTS, OUTPUTS, FUNCTION_INTENTION, SQL_BODY (NULL=Direct), STAGE_NAME (NULL).

V1 lists enums in the prompt but does NOT enforce them at the schema level.

In [ ]:
CALL SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION(
    'AI_STUDIO_DEMO.PUBLIC.CLASSIFY_SUPPORT_TICKET',
    'claude-sonnet-4-6',
    $$You are an expert enterprise support ticket classifier. Analyze B2B support tickets and classify them.

Division: Industrial Adhesives & Tapes, Safety & Industrial, Electronics & Energy, Healthcare, Transportation & Electronics, Consumer
Priority: P1-Critical (line stopped/safety), P2-High (quality/VP escalation), P3-Standard (tech inquiry/RMA), P4-Low (general/catalog)
Issue Types: Product Defect / Failure, Technical Specification Inquiry, Custom Formulation Request, Supply Chain / Delivery, Regulatory / Compliance, Warranty / RMA, Pricing / Contract Dispute, Sample Request, Application Engineering Support
Customer Segments: OEM, Distributor, Converter, End User, Government/Defense
Escalation Indicators: production_line_down, safety_incident, regulatory_deadline, executive_involvement, contractual_risk
Routing Queue: DIVISION_CODE-FUNCTION-SECTOR (IAT,SI,EE,HC,TE,CON + AppEng,QA,TechServ,Regulatory,Sales,Warranty,Logistics)
Response Templates: ACK-P1-DEFECT, ACK-P1-SAFETY, ACK-P2-QUALITY, ACK-P2-REGULATORY, ACK-SPEC-REQUEST, ACK-SAMPLE-REQUEST, ACK-RMA, ACK-WARRANTY, ACK-APPENG, ACK-GENERAL, ACK-PRICING$$,
    $$Classify the following enterprise support ticket:

{TICKET_TEXT}$$,
    PARSE_JSON('[{"name":"TICKET_TEXT","sql_type":"VARCHAR"}]'),
    PARSE_JSON('[{"name":"division","json_type":"string","description":"Business division"},{"name":"issue_type","json_type":"string","description":"Issue category"},{"name":"priority","json_type":"string","description":"P1-P4"},{"name":"priority_reasoning","json_type":"string","description":"Justification"},{"name":"routing_queue","json_type":"string","description":"Team queue"},{"name":"product_family","json_type":"string","description":"Product family"},{"name":"customer_segment","json_type":"string","description":"Customer type"},{"name":"sla_flag","json_type":"boolean","description":"SLA indicator"},{"name":"escalation_indicators","json_type":"array","description":"Escalation flags"},{"name":"suggested_response_template","json_type":"string","description":"Template ID"}]'),
    'Classify enterprise B2B support tickets by division, priority, and routing queue',
    NULL,
    NULL
);

### Test V1

In [ ]:
SELECT 
  c:division::VARCHAR AS division,
  c:issue_type::VARCHAR AS issue_type,
  c:priority::VARCHAR AS priority,
  c:routing_queue::VARCHAR AS routing_queue
FROM (
  SELECT CLASSIFY_SUPPORT_TICKET(TICKET_TEXT) AS c
  FROM SUPPORT_TICKETS WHERE TICKET_ID = 'TKT-2026-00142'
);

### V1 Observation

The issue_type field generates verbose labels instead of exact enum values. This is the key problem V2 solves.

---

## V2: Improved Enum Adherence

Changes: stronger "MUST select EXACTLY" language, few-shot example, tighter output descriptions.

Note: LLMs are probabilistic. Results vary between runs. Focus on directional improvement.

In [ ]:
CALL SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION(
    'AI_STUDIO_DEMO.PUBLIC.CLASSIFY_SUPPORT_TICKET_V2',
    'claude-sonnet-4-6',
    $$You are an enterprise support ticket classifier.

CRITICAL: Select values EXACTLY from allowed enums. Do NOT paraphrase.

division: "Industrial Adhesives & Tapes", "Safety & Industrial", "Electronics & Energy", "Healthcare", "Transportation & Electronics", "Consumer"
issue_type: "Product Defect / Failure", "Technical Specification Inquiry", "Custom Formulation Request", "Supply Chain / Delivery", "Regulatory / Compliance", "Warranty / RMA", "Pricing / Contract Dispute", "Sample Request", "Application Engineering Support"
priority: "P1-Critical" (line stopped/safety/<72h), "P2-High" (quality/VP/SLA), "P3-Standard" (inquiry/RMA/samples), "P4-Low" (general/catalog)
customer_segment: "OEM", "Distributor", "Converter", "End User", "Government/Defense"
escalation_indicators: "production_line_down", "safety_incident", "regulatory_deadline", "executive_involvement", "contractual_risk"
routing_queue: DIVISION_CODE-FUNCTION-SECTOR (IAT,SI,EE,HC,TE,CON + AppEng,QA,TechServ,Regulatory,Sales,Warranty,Logistics)
suggested_response_template: ACK-P1-DEFECT, ACK-P1-SAFETY, ACK-P2-QUALITY, ACK-P2-REGULATORY, ACK-SPEC-REQUEST, ACK-SAMPLE-REQUEST, ACK-RMA, ACK-WARRANTY, ACK-APPENG, ACK-GENERAL, ACK-PRICING

EXAMPLE:
Input: "Production line down 4 hours, adhesive failure. VP involved."
Output: {"division":"Industrial Adhesives & Tapes","issue_type":"Product Defect / Failure","priority":"P1-Critical","priority_reasoning":"Line down.","routing_queue":"IAT-AppEng-Automotive","product_family":"Structural Adhesives","customer_segment":"OEM","sla_flag":true,"escalation_indicators":["production_line_down","executive_involvement"],"suggested_response_template":"ACK-P1-DEFECT"}$$,
    $$Classify the following enterprise support ticket:

{TICKET_TEXT}$$,
    PARSE_JSON('[{"name":"TICKET_TEXT","sql_type":"VARCHAR"}]'),
    PARSE_JSON('[{"name":"division","json_type":"string","description":"Exact enum value"},{"name":"issue_type","json_type":"string","description":"Exact enum value"},{"name":"priority","json_type":"string","description":"P1-Critical/P2-High/P3-Standard/P4-Low"},{"name":"priority_reasoning","json_type":"string","description":"1-2 sentence justification"},{"name":"routing_queue","json_type":"string","description":"CODE-FUNCTION-SECTOR"},{"name":"product_family","json_type":"string","description":"Product family"},{"name":"customer_segment","json_type":"string","description":"Exact enum value"},{"name":"sla_flag","json_type":"boolean","description":"SLA indicator"},{"name":"escalation_indicators","json_type":"array","description":"From allowed list only"},{"name":"suggested_response_template","json_type":"string","description":"ACK template from allowed list"}]'),
    'Classify enterprise B2B support tickets with strict enum adherence',
    NULL,
    NULL
);

### Test V2

In [ ]:
SELECT 
  c:division::VARCHAR AS division,
  c:issue_type::VARCHAR AS issue_type,
  c:priority::VARCHAR AS priority,
  c:routing_queue::VARCHAR AS routing_queue,
  c:escalation_indicators AS escalation_indicators
FROM (
  SELECT CLASSIFY_SUPPORT_TICKET_V2(TICKET_TEXT) AS c
  FROM SUPPORT_TICKETS WHERE TICKET_ID = 'TKT-2026-00142'
);

### Key Improvement

V2 returns exact enum values. The stronger prompt + output descriptions drive adherence. Both functions are now tagged and tracked by AI Function Studio for use with EVALUATE and OPTIMIZE.

---
## Complete

Proceed to **Notebook 3: Evaluation & Iteration**.

In [ ]:
MERGE INTO DEMO_STATE AS t
USING (SELECT 'function_creation' AS NOTEBOOK_ID, CURRENT_TIMESTAMP() AS COMPLETED_AT) AS s
ON t.NOTEBOOK_ID = s.NOTEBOOK_ID
WHEN MATCHED THEN UPDATE SET COMPLETED_AT = s.COMPLETED_AT
WHEN NOT MATCHED THEN INSERT (NOTEBOOK_ID, COMPLETED_AT) VALUES (s.NOTEBOOK_ID, s.COMPLETED_AT);